# Batch Deployment Paradigm for LLM-Based Systems

Batch deployment represents the simplest and most scalable strategy for integrating an AI model or Compound AI Agent into a production environment when real-time interactivity is not required. It focuses on efficiency and high throughput over immediate responsiveness.

### I. Core Principles and Mechanics

* **Asynchronous Prediction Generation:** The fundamental idea is to generate predictions or **completions** (in the case of Generative AI) on a **regular, pre-defined schedule** (e.g., every hour, daily, weekly), rather than in response to an immediate user request.
* **Materialization to Storage Layer:** The output from the AI system is immediately **persisted** (or "materialized") to a dedicated storage layer (e.g., a data warehouse, a structured database, or a vector store).
* **Downstream Utility:** These materialized outputs are then leveraged for **ad-hoc Business Intelligence (BI)**, analytical reports, or served indirectly to end-users via non-interactive channels.
* **Latency Threshold Heuristic:** Batch processing is ideal when the required response time is flexible. A practical rule of thumb suggests this paradigm is appropriate if the pace at which new data arrives or changes is **slower than a 30-minute interval**, allowing ample time to schedule and complete the batch job.

### II. Ideal Use Cases and Advantages

* **High Throughput and Data Volume:** Batch deployment is optimized for handling **large volumes of data** efficiently by grouping requests together (batching), which maximizes the utilization of underlying hardware (e.g., GPUs).
* **Non-Immediate Response Requirement:** It is the correct paradigm for tasks where instant feedback is unnecessary.
* **Generative AI Example: Automated Document Processing:** Automating legal research, where a system **periodically ingests** updated legal databases (cases, contracts), uses an LLM to generate **summaries** or **digested newsletters**, and then persists the derived information for later consumption.
* **Implementation Ease (With a Caveat):** Conceptually, batch is the simplest deployment strategy. The engineering focus shifts from managing high-concurrency, low-latency infrastructure to optimizing data pipelines and resource scheduling.

### III. Limitations and Engineering Challenges

* **High Latency:** By definition, batch deployment is unsuitable for scenarios requiring instantaneous, synchronous interaction (e.g., chatbots). The latency is measured in the gap between job executions (e.g., hours).
* **Stale Data Risk:** A primary concern is that the predictions may be based on **stale data**. Engineers must implement **data monitoring jobs** to ensure that the scheduled batch process is running on the freshest or most current available data.
* **Cost Management for LLMs (The "Not That Cheap" Factor):** While generally considered "cheap" for traditional ML, batch inference for **Large Language Models (LLMs)** can become very expensive due to the significant computational resources (GPU hours) required for token generation at scale. Optimized batching and efficient model serving are critical to manage these costs.
* **LLM Friction:** Achieving efficient, high-throughput batch inference for LLMs often requires specialized tooling and techniques beyond simple API calls, such as model serving frameworks designed for maximum GPU utilization.

# Batch Deployment Workflow for Small to Medium Language Models

A typical batch deployment workflow for LLMs focuses on efficiently integrating the model into a data pipeline to process large datasets on a scheduled basis. While applicable to large models, this often requires access to sufficiently large **GPUs** and specialized infrastructure.

### I. Model Sourcing and Centralized Registration

The initial phase involves obtaining the appropriate language model and registering it in a central governance location, such as the **Unity Catalog Model Registry**.

* **Model Sourcing Options:** AI Engineers have three primary sources for acquiring the base model:
    * **Do-It-Yourself (DIY) Models:** Custom models built, pre-trained, or fine-tuned internally by the engineering team.
    * **Hugging Face Model Hub Integration:** Leveraging the vast ecosystem of open-source models, typically stored in the **Hugging Face Transformer** format. This involves downloading the model weights and logging them into the MLOps system.
    * **Databricks Marketplace:** Utilizing pre-vetted, curated models offered within the platform's marketplace, including those sourced from Hugging Face or shared by other platform users.
* **Registration into Unity Catalog:** Regardless of the source, the model is standardized by being logged into the **Unity Catalog Registry** using **MLflow** with the appropriate **Transformer flavor** (or Python flavor for DIY solutions). This registration is performed under a specific **Catalog and Schema** to ensure proper governance, versioning, and access control.
* **Model Preparation:** Before registration, the model may be used "as is" or undergo a critical preparation phase:
    * **Fine-Tuning:** The most common preparation step is fine-tuning the model on proprietary, in-house data to specialize its knowledge or optimize its performance for the target batch task.
    * **Simple Logging:** For models used directly, the process is streamlined: download the weights and log them to MLflow, ensuring all necessary dependencies are captured.

### II. Batch Inference Execution Strategies

Once the model is registered, the batch job can be executed on new data flowing into the system. The execution phase offers multiple, scalable options based on the user's skillset and the required concurrency.

* **Option 1: Single-Node Inference with Spark UDFs (for Simplicity and Small Scale):**
    * **Mechanism:** Running the batch inference using the model's standard **predict() method** wrapped inside a **Spark User-Defined Function (UDF)**.
    * **Scope:** The model is typically loaded onto a single node. While scalable to a small degree, this approach may be less efficient for very large datasets as it doesn't fully exploit the distributed nature of the cluster for model loading or execution.
    * **Ideal Use Case:** Rapid prototyping or processing small-to-medium datasets where cluster-wide distribution of the model is not strictly necessary.

* **Option 2: Multi-Node Distributed Inference (for Scalability):**
    * **Mechanism:** Utilizing distributed computing techniques (e.g., Spark with appropriate cluster configurations) to load copies of the model or its components across multiple worker nodes.
    * **Scope:** Allows for highly efficient processing of massive datasets by distributing the inference workload. This method is essential when the inference time per record is significant or the data volume is extremely high.

* **Option 3: AI Query Functions (for SQL-Centric Workflows):**
    * **Mechanism:** Leveraging platform-specific intelligence features, such as Databricks' **AI Query** functionality. This allows users to invoke registered foundation models directly from the **SQL language**.
    * **Scope:** Designed for **SQL users** (Data Analysts, Data Engineers) who may not be comfortable with Python-based ML code. It abstracts the complexity of model loading and execution, enabling them to run large-scale batch inferences using familiar SQL syntax on the platform's foundation models.
    * **Efficiency:** These platform-native functions are optimized for distributed inference, making them an efficient way to execute batch jobs without writing explicit MLOps code.

# AI Query: Scalable Batch Inference for Foundation Models


The **AI Query** feature is an advanced platform primitive designed to seamlessly integrate Large Language Model (LLM) inference into existing data workflows, particularly for large-scale **batch processing** needs. It abstracts away the complexities of deploying and managing LLM serving infrastructure.

### I. Technical Definition and Core Functionality

SELECT AI_QUERY(   
  "FONDATION_MODEL",   
  CONCAT("PROMPT BASED ON SOME COLUMN", COLUMN_NAME   
  ")   
) AS OUTPUT_COLUMN_NAME FROM TABLE_NAME   

* **SQL-Native Batch Invocation:** **AI Query** is exposed as a **SQL method** available within the Databricks Intelligence Platform. This allows data engineers and analysts to invoke Foundation Models directly from standard SQL queries, integrating Generative AI into traditional data pipelines without writing extensive Python code.
* **Foundation Model Execution:** The method facilitates **batch invocation** on pre-deployed and optimized **Foundation Models** (such as Databricks' own **DBRX Instruct** model) that are served via the platform's dedicated **Foundation Models API**.
* **Automatic Output Parsing:** The feature handles the complexities of LLM outputs by automatically parsing the **completions** (the generated text) and returning them as a structured result within the SQL environment, simplifying the downstream consumption of the model's output.

### II. Implementation and Use Case (Batching Requests)

* **Workflow Example:** An engineer can define a SQL query that selects data (e.g., a column of customer reviews) from a table and applies the `AI_QUERY()` function to it.
* **Prompt Structuring:** Within the function call, the engineer defines the AI task, which typically involves two components:
    * **System Prompt:** A high-level, static instruction that sets the model's role and constraints for the entire batch job (e.g., "Based on the review, determine if the customer is happy or not and respond to ensure satisfaction.").
    * **Input Column:** The dynamic text column from the database table (e.g., the `review_column` from a `reviews` table) that is fed into the model for each row's completion.
* **Efficiency of Batching:** The system efficiently **batches these individual requests** to the underlying LLM API endpoint, significantly improving the throughput and cost efficiency compared to making a high volume of individual, non-batched API calls.
* **Primary Goal:** The function’s design is to enable **large-scale, scheduled data enrichment**—for instance, automatically analyzing and generating tailored responses for millions of customer reviews stored in a data warehouse.

# Engineering Considerations for Batch Deployment of Large Language Models (LLMs)

While batch deployment is conceptually simple, executing it efficiently for multi-billion parameter LLMs presents significant technical hurdles related to hardware provisioning, cost optimization, and parallel processing.

### I. Hardware and Memory Constraints

* **Requirement for High-Capacity GPUs:** The primary friction point for running self-managed batch inference on large LLMs is the mandatory need for specialized **GPUs with substantial VRAM (GPU memory)**. Smaller models can run on CPUs or smaller GPUs, but performant LLMs require dedicated high-end hardware.
* **VRAM Sizing Requirement (Example Calculation):** The VRAM footprint scales directly with the number of parameters and the precision used to store their weights.
    * For a **10 Billion Parameter Model** using **32-bit floating point precision (4 bytes per weight)**:
        $$\text{VRAM Footprint} = 10 \times 10^9 \text{ parameters} \times 4 \text{ bytes/parameter} = 40 \text{ Gigabytes (GB)}.$$
    * This memory requirement necessitates access to high-end accelerators like the **NVIDIA A100** or **V100**, which are costly and may have procurement or availability constraints.
* **Economic Maximization:** Engineers must prioritize maximizing the **GPU utilization** rate to optimize the economics of the batch job. If the expensive hardware sits idle, the cost per completion becomes prohibitively high.

### II. Parallelization and Distributed Computing Complexity

* **Single-Node Multi-GPU Parallelization:** While frameworks exist to facilitate running inference across multiple GPUs on a single server (single node), this requires careful memory and task partitioning management.
* **Multi-Node Multi-GPU Cluster Parallelization:** Distributing a single LLM inference job across an entire cluster composed of multiple nodes, each with multiple GPUs, is a **non-trivial engineering task**. This requires sophisticated distributed computing orchestration to manage data partitioning, model parallelism, and communication overhead.
* **Active Research Area:** The problem of achieving efficient, high-throughput, and scalable batch inference on large, distributed LLMs remains an **active and complex area of research** and engineering development.

### III. Mitigating Complexity with Specialized Tooling

* **Open-Source Integration Options:** To manage the complexity of parallel execution and memory efficiency, engineers can leverage specialized open-source libraries:
    * **TensorRT-LLM:** A library designed by NVIDIA to optimize and accelerate the inference performance of LLMs on NVIDIA GPUs.
    * **vLLM:** A high-throughput and low-latency inference engine that employs techniques like PagedAttention to enhance batching and memory management.
    * **Ray on Spark:** Provides a framework for building scalable distributed applications, potentially simplifying the multi-node coordination.
* **The API Paradigm Shift:** Due to the inherent complexity and operational overhead of setting up self-managed batch infrastructure, the industry trend is increasingly moving towards leveraging **Real-Time or REST APIs** for completions, even for large-scale batch jobs. By using a managed service (either internal or external), engineers trade potential per-request cost for simplified infrastructure management and guaranteed high utilization rates.

# Demo Batch Inference SLM

In [0]:
%pip install huggingface_hub transformers==4.38.2 databricks-langchain mlflow torch

In [0]:
import json
import torch

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

login(token = json.load(open("../config/config.json"))["HF_PAT"])

In [0]:
print(torch.__version__)

In [0]:
summarization_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

sumarizer = AutoModelForCausalLM.from_pretrained(
    summarization_model_id,
    torch_dtype=torch.bfloat16,
    device_map = "auto"
)